# Hardware-Aware YOLOX-Nano NAS (Standalone Colab Version)

This notebook is designed to be uploaded directly to **Google Colab** in your web browser. 
It will clone your repository, set up the proxy dataset, and run the NNI Neural Architecture Search. 

### Instructions:
1. Go to [colab.research.google.com](https://colab.research.google.com/).
2. Upload this `.ipynb` file.
3. Go to **Runtime > Change runtime type** and select **T4 GPU**.
4. Run the cells sequentially.

In [ ]:
# 1. Clone Repository and Install Base Dependencies
import os
if not os.path.exists('edge-ai-object-detection'):
    !git clone --recursive https://github.com/Bey3fendi/edge-ai-object-detection.git

%cd edge-ai-object-detection

!pip install -q torch torchvision torchaudio
!pip install -q onnx onnxruntime pytest nni loguru thop tabulate

In [ ]:
# 2. Build YOLOX
%cd YOLOX
!pip install -q -v -e .
!pip install -q cython
!pip install -q 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
%cd ..

In [ ]:
# 3. Download COCO Annotations & Create 10k Proxy Dataset
import os
os.makedirs('datasets/coco/annotations', exist_ok=True)

if not os.path.exists('datasets/coco/annotations/instances_train2017.json'):
    print("Downloading COCO annotations...")
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O annotations.zip
    !unzip -q annotations.zip -d datasets/coco/
    !rm annotations.zip

print("Generating 10k proxy dataset for rapid NAS...")
!python3 scripts/nas/create_coco_proxy.py --input datasets/coco/annotations/instances_train2017.json --output datasets/coco/annotations/instances_proxy10k.json --samples 10000

## 4. Run NNI Architecture Search
Since Colab runs in the cloud, we can't easily open `localhost:8080` to see the NNI Web UI. 
We will use `localtunnel` to create a public link to your NNI dashboard.

In [ ]:
!npm install -g localtunnel

import subprocess
import urllib
import time

# Start NNI in the background
subprocess.Popen(["nnictl", "create", "--config", "scripts/nas/nas_config.yml", "--port", "8080"])
time.sleep(5) # Give it a few seconds to boot up

# Output the password for localtunnel
print("=============================================================")
print("When you click the link below, it will ask for an IP/Password.")
print("Copy this IP and paste it into the website:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("=============================================================")

# Start localtunnel to expose port 8080
!lt --port 8080

## 5. Harvest the Best Architecture
Once NNI completes the 20 trials, you can extract the best parameters from the UI or by running the command below, and copy them back to your local WSL environment to proceed to Pruning.

In [ ]:
!nnictl experiment list
# Replace 'EXPERIMENT_ID' with your actual ID from the list above
# !nnictl experiment export EXPERIMENT_ID --filename best_nas_params.json --type json